# ⏱️ Clase 2 — Sesgo Temporal, Desbalanceo y Data Leakage (Python)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · `drdarioezequieldiaz@gmail.com`

**Fecha de referencia:** Jueves 16 de abril de 2026

---

### 🎯 Objetivos del notebook

1. **Simular** concept drift sobre un dataset crediticio introduciendo desplazamientos controlados entre períodos de desarrollo y producción.
2. **Implementar** el Population Stability Index (PSI) desde primeros principios y aplicar los umbrales del semáforo regulatorio (verde/amarillo/rojo).
3. **Visualizar** los tres tipos de drift distribucional: covariate shift, concept shift y prior probability shift.
4. **Demostrar empíricamente** el Teorema de inutilidad del accuracy bajo desbalanceo severo, usando un clasificador trivial como referencia.
5. **Aplicar** técnicas de remuestreo (SMOTE, ADASYN) y comparar contra el enfoque cost-sensitive mediante `class_weight`, usando siempre la curva Precision-Recall como métrica diagnóstica.
6. **Construir** cuatro ejemplos reproducibles de data leakage (target leakage, train-test contamination, temporal leakage, SMOTE mal aplicado) midiendo la inflación artificial de métricas que produce cada uno.
7. **Implementar** el pipeline canónico blindado contra leakage con `sklearn.pipeline.Pipeline` y `imblearn.pipeline.Pipeline`, validado con `TimeSeriesSplit`.

### 📚 Estrategia pedagógica

Retomamos el **German Credit Dataset** ya empleado en notebooks previos de la Clase 2, enriqueciéndolo con una variable temporal sintética que nos permite simular ventanas de desarrollo y producción. Sobre ese sustrato construimos tres laboratorios complementarios: en el primero inducimos drift y lo detectamos con PSI; en el segundo, acentuamos el desbalanceo natural del dataset y medimos el impacto sobre las métricas; en el tercero, construimos de manera deliberada cuatro tipos de fugas de información y cuantificamos su capacidad de engaño.

> **Referencia teórica:** Apunte de Clase 2, Parte III — *Sesgo Temporal, Desbalanceo de Clases y Data Leakage* (Díaz, 2026).

---

## 1. Configuración del entorno

El stack necesario se organiza en cinco bloques funcionales:

| Bloque | Librerías | Función |
|---|---|---|
| **Base** | `pandas`, `numpy`, `matplotlib`, `seaborn` | Manipulación y visualización |
| **Modelado** | `sklearn` (linear_model, ensemble, metrics, model_selection, preprocessing, pipeline) | Algoritmos, métricas, splitters, pipelines |
| **Remuestreo** | `imbalanced-learn` (`SMOTE`, `ADASYN`, `Pipeline`) | Técnicas de rebalanceo de clases |
| **Estadística** | `scipy.stats` | Divergencia de Kullback-Leibler, tests |
| **Dataset** | `sklearn.datasets.fetch_openml` | German Credit (Statlog) |

La única dependencia no estándar es `imbalanced-learn`, que se instala condicionalmente.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CONFIGURACIÓN DEL ENTORNO
# ══════════════════════════════════════════════════════════════

import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Instala pkg si no puede importarse. Mantiene Colab limpio."""
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"Instalando {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("imbalanced-learn", "imblearn")

# --- Importaciones ---------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      TimeSeriesSplit, cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              confusion_matrix, classification_report,
                              average_precision_score, fbeta_score)

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

from scipy import stats

# --- Configuración estética -----------------------------------------
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["figure.dpi"] = 100

# Paleta del curso (Midnight Executive)
COL_NAVY   = "#1A2744"
COL_TEAL   = "#0C7C8A"
COL_AMBER  = "#F59E0B"
COL_RED    = "#E76F51"
COL_PURPLE = "#7C3AED"
COL_GREEN  = "#10B981"

# Semáforo PSI
SEM_VERDE    = "#10B981"
SEM_AMARILLO = "#F59E0B"
SEM_ROJO     = "#E76F51"

# Reproducibilidad global
RNG_SEED = 2026
np.random.seed(RNG_SEED)

print("Entorno configurado correctamente")
print(f"   pandas {pd.__version__} . numpy {np.__version__}")
import sklearn, imblearn
print(f"   sklearn {sklearn.__version__} . imblearn {imblearn.__version__}")

## 2. Carga del dataset y enriquecimiento temporal

Trabajamos sobre el **German Credit Dataset (Statlog)**, ya familiar. Para habilitar el análisis de sesgo temporal incorporamos una variable `periodo` sintética que distribuye las 1000 observaciones en cuatro trimestres consecutivos: `2024Q1`, `2024Q2`, `2024Q3` y `2024Q4`. Los dos primeros trimestres constituirán nuestro **período de desarrollo** y los dos últimos, el **período de producción** donde mediremos el drift.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CARGA DEL DATASET + ENRIQUECIMIENTO TEMPORAL
# ══════════════════════════════════════════════════════════════

german = fetch_openml(name="credit-g", version=1, as_frame=True, parser="auto")
df = german.frame.copy()

# La variable respuesta viene como categórica: codificamos a binaria
# siguiendo la convención de scoring crediticio: 1 = default (mal pagador).
df["default"] = (df["class"] == "bad").astype(int)
df = df.drop(columns=["class"])

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Tasa de default observada: {df['default'].mean():.1%}")
print(f"Valores faltantes: {df.isna().sum().sum()}")

# --- Enriquecimiento temporal ----------------------------------------
rng = np.random.default_rng(RNG_SEED)

# Asignamos cada registro a uno de cuatro trimestres, con probabilidades
# aproximadamente uniformes.
periodos = ["2024Q1", "2024Q2", "2024Q3", "2024Q4"]
df["periodo"] = rng.choice(periodos, size=len(df), p=[0.26, 0.26, 0.24, 0.24])

# Ordenamos por período para respetar la estructura temporal en las
# ventanas de validación posteriores.
df = df.sort_values("periodo").reset_index(drop=True)

print("\nDistribución por período:")
print(df["periodo"].value_counts().sort_index().to_string())

### 2.1 Preparación de features

Separamos los predictores del target y codificamos las variables categóricas con `pd.get_dummies` (one-hot), estrategia suficiente para los modelos lineales y basados en árboles que emplearemos. Conservamos la columna `periodo` **fuera** de la matriz de features: es metadato, no predictor.

In [ ]:
# --- Separación predictores / target ---------------------------------
y = df["default"].values
periodo_col = df["periodo"].copy()
X = df.drop(columns=["default", "periodo"])

# --- Codificación one-hot de categóricas -----------------------------
# drop_first=True para evitar la trampa de la variable ficticia en modelos lineales
X_enc = pd.get_dummies(X, drop_first=True)
X_enc = X_enc.astype(float)

print(f"Matriz de features: {X_enc.shape[0]} filas x {X_enc.shape[1]} columnas")
print(f"Nombres de las primeras 8 columnas:")
for c in X_enc.columns[:8]:
    print(f"   - {c}")
print(f"   ...")

### 📝 Interpretación

Partimos de un dataset íntegro, con 1000 observaciones repartidas aproximadamente en partes iguales a lo largo de cuatro trimestres, y una tasa de default global del **30%**. Esta tasa, si bien es razonable académicamente, es elevada para un portafolio real: los bancos reportan típicamente tasas de default entre el 5% y el 15% en préstamos personales, según la tabla de la sección 3 del apunte. En la Parte II del notebook usaremos esta tasa natural para explorar la interacción entre desbalanceo leve y desbalanceo severo inducido.

---

# 🌀 PARTE I — SESGO TEMPORAL (CONCEPT DRIFT)

> *"Las relaciones estadísticas descubiertas durante el entrenamiento permanecen estables cuando el modelo se despliega en producción. En el dominio financiero, este supuesto resulta sistemáticamente violado."* (Apunte, §1)

---

## 3. Simulación controlada de los tres tipos de drift

El apunte identifica tres tipos de drift según el componente de la distribución conjunta $P(X, Y)$ afectado:

$$P_t(X, Y) = P_t(Y \mid X) \cdot P_t(X)$$

- **Covariate shift**: cambia $P_t(X)$ con $P(Y|X)$ estable. Ejemplo: devaluación que altera montos nominales sin cambiar la relación riesgo-monto.
- **Concept shift**: cambia $P_t(Y \mid X)$. Ejemplo: recesión que deteriora el perfil de pagadores para iguales atributos.
- **Prior probability shift**: cambia $P_t(Y)$. Ejemplo: la tasa global de default sube del 3% al 12% en crisis sistémica.

Para hacer la demostración didácticamente limpia, **inducimos cada tipo de shift de manera controlada** sobre la variable `credit_amount` y la tasa de default, creando distribuciones `dev` y `prod` que podemos comparar.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. SIMULACIÓN DE LOS TRES TIPOS DE DRIFT
# ══════════════════════════════════════════════════════════════

rng = np.random.default_rng(RNG_SEED + 10)

# Población base: una distribución empírica sobre credit_amount
amount_base = df["credit_amount"].values
n_dev  = 600
n_prod = 400

# --- (A) Covariate shift: desplazamos la media de la predictora --------
# Simulamos devaluación: los montos nominales se duplican
dev_covariate  = rng.choice(amount_base, size=n_dev, replace=True)
prod_covariate = rng.choice(amount_base, size=n_prod, replace=True) * 2.0

# --- (B) Prior probability shift: cambia la tasa de default ----------
# Dev: 10% default. Prod: 40% default (crisis sistémica)
dev_prior  = rng.binomial(1, 0.10, size=n_dev)
prod_prior = rng.binomial(1, 0.40, size=n_prod)

# --- (C) Concept shift: mismo X, distinto P(Y|X) ---------------------
# Simulamos score de riesgo dependiente del monto con una pendiente
# que cambia de dev a prod.
x_dev = rng.uniform(500, 15000, size=n_dev)
x_prod = rng.uniform(500, 15000, size=n_prod)

def prob_default(x, pendiente):
    """Logística con intercepto -2 y pendiente parametrizable."""
    logit = -2 + pendiente * (x - 7500) / 7500
    return 1 / (1 + np.exp(-logit))

dev_concept  = rng.binomial(1, prob_default(x_dev,  pendiente=0.3))
prod_concept = rng.binomial(1, prob_default(x_prod, pendiente=1.8))

print(f"Tres escenarios de drift construidos.")
print(f"Dev samples: {n_dev} | Prod samples: {n_prod}")
print(f"\nResumen:")
print(f"   Covariate: media_dev = {dev_covariate.mean():.0f},  "
      f"media_prod = {prod_covariate.mean():.0f}")
print(f"   Prior    : tasa_dev = {dev_prior.mean():.1%}, "
      f"tasa_prod = {prod_prior.mean():.1%}")
print(f"   Concept  : tasa_dev = {dev_concept.mean():.1%}, "
      f"tasa_prod = {prod_concept.mean():.1%}")

In [ ]:
# --- Visualización comparativa de los tres tipos de drift ------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (A) Covariate shift
axes[0].hist(dev_covariate, bins=40, alpha=0.6, color=COL_TEAL,
             label="Desarrollo", edgecolor="white")
axes[0].hist(prod_covariate, bins=40, alpha=0.6, color=COL_RED,
             label="Producción", edgecolor="white")
axes[0].set_title("(A) Covariate shift\nCambia P(X), P(Y|X) estable",
                  color=COL_NAVY, fontweight="bold")
axes[0].set_xlabel("credit_amount"); axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# (B) Prior probability shift
bars1 = axes[1].bar([0, 1], [1 - dev_prior.mean(), dev_prior.mean()],
                    width=0.35, color=COL_TEAL, alpha=0.8, label="Desarrollo")
bars2 = axes[1].bar([0.4, 1.4], [1 - prod_prior.mean(), prod_prior.mean()],
                    width=0.35, color=COL_RED, alpha=0.8, label="Producción")
axes[1].set_xticks([0.2, 1.2])
axes[1].set_xticklabels(["No default (Y=0)", "Default (Y=1)"])
axes[1].set_title("(B) Prior probability shift\nCambia P(Y)",
                  color=COL_NAVY, fontweight="bold")
axes[1].set_ylabel("Proporción"); axes[1].legend()

# (C) Concept shift: plot de P(Y=1|X) por escenario
x_grid = np.linspace(500, 15000, 200)
axes[2].plot(x_grid, prob_default(x_grid, 0.3), color=COL_TEAL,
             lw=2.5, label="Desarrollo (pendiente=0.3)")
axes[2].plot(x_grid, prob_default(x_grid, 1.8), color=COL_RED,
             lw=2.5, label="Producción (pendiente=1.8)")
axes[2].set_title("(C) Concept shift\nCambia P(Y|X), P(X) estable",
                  color=COL_NAVY, fontweight="bold")
axes[2].set_xlabel("credit_amount"); axes[2].set_ylabel("P(default | X)")
axes[2].legend()

plt.tight_layout(); plt.show()

### 📝 Interpretación

Los tres paneles ilustran fenómenos estructuralmente distintos:

- **Panel A**: la nube de registros de producción está **desplazada** respecto de la de desarrollo, pero la relación subyacente entre monto y riesgo permanece (solo cambió la escala). Es el efecto típico de una **devaluación**: $P(X)$ se traslada, $P(Y|X)$ no.
- **Panel B**: las proporciones de clase cambian drásticamente. La **crisis** eleva la tasa de default global del 10% al 40% sin alterar los mecanismos causales. Es un shift de $P(Y)$.
- **Panel C**: para un mismo valor de `credit_amount`, la probabilidad de default cambió. La curva logística se **inclinó**: montos que en desarrollo tenían 30% de probabilidad de default, en producción tienen 65%. Es el drift más peligroso porque el modelo entrenado en desarrollo sistemáticamente subestima el riesgo de los casos de mayor monto.

## 4. Population Stability Index (PSI)

La definición formal del apunte:

$$\text{PSI} = \sum_{b=1}^{B} \left(p_b^{\text{prod}} - p_b^{\text{dev}}\right) \cdot \ln\left(\frac{p_b^{\text{prod}}}{p_b^{\text{dev}}}\right)$$

donde las distribuciones de desarrollo y producción se discretizan en $B$ intervalos (típicamente deciles). La Proposición 3.2 conecta el PSI con la divergencia de Kullback-Leibler: es la suma simétrica de $D_{KL}(P_{\text{prod}} \| P_{\text{dev}}) + D_{KL}(P_{\text{dev}} \| P_{\text{prod}})$.

Los **umbrales regulatorios estándar** en la industria financiera:

| Rango | Semáforo | Interpretación |
|---|---|---|
| $\text{PSI} < 0.10$ | 🟢 Verde | Distribución estable |
| $0.10 \leq \text{PSI} < 0.20$ | 🟡 Amarillo | Cambio moderado. Investigar |
| $\text{PSI} \geq 0.20$ | 🔴 Rojo | Inestabilidad severa. Recalibrar |

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. IMPLEMENTACIÓN DEL PSI
# ══════════════════════════════════════════════════════════════

def calcular_psi(dev, prod, n_bins=10, eps=1e-6):
    """
    Calcula el Population Stability Index entre dos distribuciones.

    Parámetros
    ----------
    dev, prod : array-like
        Muestras de la distribución de desarrollo y producción.
    n_bins : int
        Número de intervalos de discretización (default 10 = deciles).
    eps : float
        Epsilon aditivo para evitar log(0) en bins vacíos.

    Retorna
    -------
    psi_total : float
        PSI agregado.
    detalle : pd.DataFrame
        Contribución de cada bin.
    """
    dev = np.asarray(dev, dtype=float)
    prod = np.asarray(prod, dtype=float)

    # Detección de variable discreta de pocos valores únicos:
    # en ese caso usamos bins centrados en cada valor en lugar de
    # cuantiles, que colapsan cuando la variable es binaria o casi.
    valores_unicos_dev = np.unique(dev)
    if len(valores_unicos_dev) <= n_bins:
        # Bins centrados: un bin por valor único observado
        vals = np.sort(valores_unicos_dev)
        bin_edges = np.concatenate([
            [vals[0] - 0.5],
            (vals[:-1] + vals[1:]) / 2,
            [vals[-1] + 0.5]
        ])
    else:
        # Caso continuo: bins por cuantiles sobre desarrollo
        bin_edges = np.quantile(dev, np.linspace(0, 1, n_bins + 1))
        bin_edges[0]  -= 1e-6
        bin_edges[-1] += 1e-6

    n_bins_real = len(bin_edges) - 1

    # Discretización
    dev_counts  = np.histogram(dev,  bins=bin_edges)[0]
    prod_counts = np.histogram(prod, bins=bin_edges)[0]

    # Proporciones + epsilon anti log(0)
    p_dev  = (dev_counts  + eps) / (dev_counts.sum()  + n_bins_real * eps)
    p_prod = (prod_counts + eps) / (prod_counts.sum() + n_bins_real * eps)

    # PSI por bin y total
    contrib = (p_prod - p_dev) * np.log(p_prod / p_dev)
    psi_total = contrib.sum()

    detalle = pd.DataFrame({
        "bin": range(1, n_bins_real + 1),
        "bin_min": bin_edges[:-1],
        "bin_max": bin_edges[1:],
        "p_dev": p_dev,
        "p_prod": p_prod,
        "contribucion_psi": contrib
    })
    return psi_total, detalle


def clasificar_psi(psi):
    """Traducción al semáforo regulatorio."""
    if psi < 0.10:
        return "Verde", SEM_VERDE
    elif psi < 0.20:
        return "Amarillo", SEM_AMARILLO
    return "Rojo", SEM_ROJO


# --- Cálculo sobre el escenario de covariate shift -------------------
psi_cov, detalle_cov = calcular_psi(dev_covariate, prod_covariate, n_bins=10)
semaforo, _ = clasificar_psi(psi_cov)
print(f"Covariate shift -> PSI = {psi_cov:.4f} -> Semaforo: {semaforo}")

print("\nContribución por bin (covariate shift):")
print(detalle_cov[["bin", "p_dev", "p_prod", "contribucion_psi"]]
      .round(4).to_string(index=False))

In [ ]:
# --- PSI sobre los tres escenarios y visualización -------------------
escenarios = {
    "Covariate shift (montos)":   (dev_covariate, prod_covariate),
    "Prior shift (tasa default)": (dev_prior.astype(float), prod_prior.astype(float)),
    "Concept shift (P(Y|X))":     (dev_concept.astype(float), prod_concept.astype(float)),
}

resumen_psi = []
for nombre, (d, p) in escenarios.items():
    # Para variables binarias usamos n_bins=2
    n_bins = 2 if set(np.unique(d)) <= {0, 1} else 10
    psi_val, _ = calcular_psi(d, p, n_bins=n_bins)
    sem, color = clasificar_psi(psi_val)
    resumen_psi.append({
        "Escenario": nombre,
        "PSI": round(psi_val, 4),
        "Semaforo": sem,
        "Color": color
    })

df_psi = pd.DataFrame(resumen_psi)
print("Sintesis del PSI por escenario:\n")
print(df_psi[["Escenario", "PSI", "Semaforo"]].to_string(index=False))

# --- Gráfico: PSI por escenario con semáforo -------------------------
fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.barh(df_psi["Escenario"], df_psi["PSI"],
               color=df_psi["Color"], alpha=0.85, edgecolor="white")
ax.axvline(0.10, color=COL_NAVY, linestyle="--", lw=1, alpha=0.5)
ax.axvline(0.20, color=COL_NAVY, linestyle="--", lw=1, alpha=0.5)
ax.text(0.10, -0.5, "0.10", ha="center", fontsize=9, color=COL_NAVY)
ax.text(0.20, -0.5, "0.20", ha="center", fontsize=9, color=COL_NAVY)
for i, (b, v) in enumerate(zip(bars, df_psi["PSI"])):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10,
            fontweight="bold")
ax.set_xlabel("PSI")
ax.set_title("Population Stability Index por escenario de drift",
             color=COL_NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

### 📝 Interpretación

Las tres barras ilustran magnitudes distintas de inestabilidad. Los umbrales de 0.10 y 0.20 (líneas punteadas) demarcan los tres regímenes:

- El **prior shift** produce el PSI más alto porque las proporciones de clase binaria son extremadamente sensibles a cambios: pasar del 10% al 40% de default es un desplazamiento distribucional masivo en términos de Kullback-Leibler.
- El **covariate shift** con duplicación del monto también entra holgadamente en zona roja. Aunque el *shape* de la distribución se preserva, la traslación masiva activa la alarma.
- El **concept shift** se detecta solo observando la variable respuesta (o el score del modelo). Si nuestro monitoreo midiera únicamente las distribuciones de features, este drift pasaría inadvertido. Por eso el monitoreo integral debe abarcar **tanto features como predicciones y tasas reales**.

> ⚠️ **Lectura crítica del PSI**: es una medida **comparativa**, no causal. PSI alto indica que *algo* cambió; diagnosticar *qué* cambió requiere análisis adicional, típicamente descomposición por feature y por decil.

## 5. Validación temporal: `TimeSeriesSplit` vs `KFold`

El $k$-fold cross-validation estándar **no respeta la estructura temporal**. Un fold de validación puede contener observaciones anteriores a las del fold de entrenamiento, generando **temporal leakage**. En datos financieros esto produce métricas infladas y falsa confianza.

La alternativa canónica es `TimeSeriesSplit` de scikit-learn: expanding window donde el train siempre precede temporalmente al validation.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. TIME SERIES SPLIT vs K-FOLD ESTÁNDAR
# ══════════════════════════════════════════════════════════════

# Trabajamos sobre el dataset original con periodo ya asignado
# Creamos orden temporal
idx_temp = np.argsort(periodo_col.values)
X_sorted = X_enc.iloc[idx_temp].reset_index(drop=True)
y_sorted = y[idx_temp]
periodo_sorted = periodo_col.iloc[idx_temp].reset_index(drop=True)

n = len(X_sorted)

# --- Visualización de los splits -------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(13, 6))

# Estrategia 1: KFold estándar (baraja el dataset)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
for i, (tr, te) in enumerate(skf.split(X_sorted, y_sorted)):
    axes[0].scatter(tr, [i]*len(tr), s=2, c=COL_TEAL, marker="s",
                    label="Train" if i == 0 else None)
    axes[0].scatter(te, [i]*len(te), s=2, c=COL_RED, marker="s",
                    label="Val" if i == 0 else None)
axes[0].set_title("KFold estandar (shuffle=True)  -  NO respeta temporalidad",
                  color=COL_NAVY, fontweight="bold")
axes[0].set_ylabel("Fold"); axes[0].set_yticks(range(5))
axes[0].legend(loc="upper right")

# Estrategia 2: TimeSeriesSplit (expanding window)
tscv = TimeSeriesSplit(n_splits=5)
for i, (tr, te) in enumerate(tscv.split(X_sorted)):
    axes[1].scatter(tr, [i]*len(tr), s=2, c=COL_TEAL, marker="s",
                    label="Train" if i == 0 else None)
    axes[1].scatter(te, [i]*len(te), s=2, c=COL_RED, marker="s",
                    label="Val" if i == 0 else None)
axes[1].set_title("TimeSeriesSplit  -  respeta la temporalidad",
                  color=COL_NAVY, fontweight="bold")
axes[1].set_xlabel("Indice temporal (orden cronologico)")
axes[1].set_ylabel("Fold"); axes[1].set_yticks(range(5))
axes[1].legend(loc="upper right")

plt.tight_layout(); plt.show()

In [ ]:
# --- Comparación numérica de AUC bajo los dos protocolos -------------
modelo = LogisticRegression(max_iter=1000, random_state=RNG_SEED)

# Con KFold barajado
auc_kfold = cross_val_score(modelo, X_sorted, y_sorted,
                             cv=StratifiedKFold(n_splits=5, shuffle=True,
                                                random_state=RNG_SEED),
                             scoring="roc_auc")

# Con TimeSeriesSplit
auc_tscv = cross_val_score(modelo, X_sorted, y_sorted,
                            cv=TimeSeriesSplit(n_splits=5),
                            scoring="roc_auc")

print("Comparacion de AUC bajo dos protocolos de validacion:\n")
print(f"   KFold barajado     : AUC = {auc_kfold.mean():.4f} +/- {auc_kfold.std():.4f}")
print(f"   TimeSeriesSplit    : AUC = {auc_tscv.mean():.4f} +/- {auc_tscv.std():.4f}")
print(f"\n   Diferencia (KFold - TSCV): {auc_kfold.mean() - auc_tscv.mean():+.4f}")

### 📝 Interpretación

En este dataset la diferencia entre ambos protocolos es modesta porque no hay drift real entre trimestres (todos los registros provienen de la misma distribución generativa del Statlog). **En datos con drift efectivo, la brecha se amplía significativamente**: el KFold barajado mezcla observaciones futuras en el train, produciendo métricas optimistas que no se reproducen cuando el modelo se despliega.

La regla operativa: **en todo dataset con estructura temporal, usar `TimeSeriesSplit` o `GroupKFold` por período, nunca `KFold` con `shuffle=True`**.

---

# ⚖️ PARTE II — DESBALANCEO DE CLASES

> *"Los eventos de interés —default crediticio, fraude, churn— constituyen, por su propia naturaleza, fenómenos minoritarios."* (Apunte, §5)

---

## 6. El ratio de desbalanceo y la inutilidad del accuracy

El **ratio de desbalanceo** (IR) es:

$$\text{IR} = \frac{n_-}{n_+}$$

El Teorema 6.1 del apunte establece que un **clasificador trivial** que predice siempre la clase mayoritaria alcanza accuracy $\frac{\text{IR}}{\text{IR}+1}$, lo que para $\text{IR}=1000$ produce 99.9% de accuracy sin detectar un solo caso positivo. Vamos a demostrarlo empíricamente.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. RATIO DE DESBALANCEO Y CLASIFICADOR TRIVIAL
# ══════════════════════════════════════════════════════════════

n_pos = (y == 1).sum()
n_neg = (y == 0).sum()
IR = n_neg / n_pos

print(f"German Credit (desbalanceo natural):")
print(f"   Clase 0 (buen pagador) : {n_neg} observaciones")
print(f"   Clase 1 (default)      : {n_pos} observaciones")
print(f"   IR = {IR:.2f}  (moderadamente desbalanceado)")

# --- Inducimos desbalanceo severo para amplificar el fenómeno --------
# Submuestreamos la clase positiva para simular un problema de fraude
# con IR ≈ 20 (similar a default de hipotecas).
rng_sub = np.random.default_rng(RNG_SEED + 20)
idx_neg_all = np.where(y == 0)[0]
idx_pos_all = np.where(y == 1)[0]
idx_pos_sub = rng_sub.choice(idx_pos_all, size=35, replace=False)
idx_severo = np.concatenate([idx_neg_all, idx_pos_sub])
rng_sub.shuffle(idx_severo)

X_sev = X_enc.iloc[idx_severo].reset_index(drop=True)
y_sev = y[idx_severo]

IR_sev = (y_sev == 0).sum() / (y_sev == 1).sum()
print(f"\nEscenario severo (inducido):")
print(f"   Clase 1: {(y_sev==1).sum()}  .  Clase 0: {(y_sev==0).sum()}")
print(f"   IR_severo = {IR_sev:.1f}")

In [ ]:
# --- Clasificador trivial vs regresión logística ---------------------
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_sev, y_sev, test_size=0.3, random_state=RNG_SEED, stratify=y_sev)

# Clasificador trivial: siempre predice la mayoritaria (default de DummyClassifier)
trivial = DummyClassifier(strategy="most_frequent", random_state=RNG_SEED)
trivial.fit(X_train, y_train)
y_pred_trivial = trivial.predict(X_test)

# Modelo honesto: regresión logística básica
logit = LogisticRegression(max_iter=1000, random_state=RNG_SEED)
logit.fit(X_train, y_train)
y_pred_logit = logit.predict(X_test)
y_proba_logit = logit.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, recall_score, precision_score
comp = pd.DataFrame({
    "Modelo": ["Clasificador trivial", "Regresion Logistica"],
    "Accuracy": [accuracy_score(y_test, y_pred_trivial),
                 accuracy_score(y_test, y_pred_logit)],
    "Recall (clase 1)": [recall_score(y_test, y_pred_trivial, zero_division=0),
                          recall_score(y_test, y_pred_logit, zero_division=0)],
    "Precision (clase 1)": [precision_score(y_test, y_pred_trivial, zero_division=0),
                             precision_score(y_test, y_pred_logit, zero_division=0)],
    "AUC": [0.500, roc_auc_score(y_test, y_proba_logit)],
})
print("Comparacion empirica del Teorema 6.1:\n")
print(comp.round(3).to_string(index=False))

### 📝 Interpretación

El clasificador trivial alcanza un accuracy superior al 95% **sin detectar ni un solo default**: su recall sobre la clase positiva es exactamente 0. La regresión logística alcanza un accuracy comparable o ligeramente inferior, pero su recall es sustantivamente mejor y su AUC supera decisivamente el 0.5 de azar. 

La lección es categórica: **bajo desbalanceo, el accuracy es una métrica engañosa que premia la pereza predictiva**. Las métricas operativas son precision, recall, F-beta y, sobre todo, AUC-ROC (relación con Gini: $\text{Gini} = 2 \cdot \text{AUC} - 1$, Proposición 6.3 del apunte) y Average Precision (área bajo la curva Precision-Recall).

## 7. Curvas ROC y Precision-Recall

La curva ROC grafica TPR vs FPR a distintos umbrales; la curva Precision-Recall contrapone precision vs recall. En escenarios de desbalanceo severo, **la curva PR suele ser más informativa** que la ROC porque se concentra en el rendimiento sobre la clase minoritaria.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. CURVAS ROC Y PRECISION-RECALL
# ══════════════════════════════════════════════════════════════

fpr, tpr, _ = roc_curve(y_test, y_proba_logit)
auc_roc = roc_auc_score(y_test, y_proba_logit)

prec, rec, _ = precision_recall_curve(y_test, y_proba_logit)
auc_pr = average_precision_score(y_test, y_proba_logit)

# Línea base de la curva PR = tasa base positiva en el test
baseline_pr = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
axes[0].plot(fpr, tpr, color=COL_TEAL, lw=2.2, label=f"Modelo (AUC = {auc_roc:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color=COL_NAVY, lw=1, label="Azar (AUC = 0.5)")
axes[0].fill_between(fpr, tpr, alpha=0.15, color=COL_TEAL)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].set_title("Curva ROC", color=COL_NAVY, fontweight="bold")
axes[0].legend()

# Curva PR
axes[1].plot(rec, prec, color=COL_PURPLE, lw=2.2, label=f"Modelo (AP = {auc_pr:.3f})")
axes[1].axhline(baseline_pr, color=COL_NAVY, linestyle="--", lw=1,
                label=f"Linea base = tasa positiva = {baseline_pr:.3f}")
axes[1].fill_between(rec, prec, alpha=0.15, color=COL_PURPLE)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Curva Precision-Recall", color=COL_NAVY, fontweight="bold")
axes[1].legend()

plt.tight_layout(); plt.show()

print(f"\nMetricas sintesis:")
print(f"   AUC-ROC              = {auc_roc:.4f}")
print(f"   Gini (2*AUC - 1)     = {2*auc_roc - 1:.4f}")
print(f"   Average Precision    = {auc_pr:.4f}")
print(f"   Tasa positiva (base) = {baseline_pr:.4f}")

### 📝 Interpretación

La curva ROC sugiere un rendimiento razonable (AUC > 0.7), pero la curva PR revela la dificultad real del problema bajo desbalanceo: incluso en los umbrales que maximizan precisión, una proporción sustantiva de positivos predichos son falsos positivos. Esta dualidad es exactamente la razón por la cual el apunte recomienda usar F-beta parametrizado según el costo del negocio:

- $\beta = 2$ para fraude (penaliza falsos negativos)
- $\beta = 0{,}5$ para scoring crediticio (penaliza falsos positivos)

## 8. Técnicas de remuestreo y cost-sensitive learning

Comparamos tres estrategias:

1. **Baseline**: regresión logística sobre datos desbalanceados, sin tratamiento.
2. **SMOTE**: oversampling sintético generando nuevos positivos mediante interpolación entre vecinos.
3. **ADASYN**: SMOTE adaptativo, con mayor densidad de sintéticos en zonas difíciles.
4. **Cost-sensitive**: `class_weight='balanced'`, ajusta pesos internos sin alterar el dataset.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. COMPARACIÓN DE ESTRATEGIAS BAJO DESBALANCEO
# ══════════════════════════════════════════════════════════════

# IMPORTANTE: el remuestreo se aplica SOLO sobre el train, NUNCA sobre el test.
# Construimos cuatro pipelines que garantizan esta regla.

# Usamos imblearn.Pipeline para que los resamplers se apliquen solo en fit
pipe_baseline = SkPipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RNG_SEED))
])

pipe_smote = ImbPipeline([
    ("scaler", StandardScaler()),
    ("resampler", SMOTE(random_state=RNG_SEED, k_neighbors=3)),
    ("clf", LogisticRegression(max_iter=1000, random_state=RNG_SEED))
])

pipe_adasyn = ImbPipeline([
    ("scaler", StandardScaler()),
    ("resampler", ADASYN(random_state=RNG_SEED, n_neighbors=3)),
    ("clf", LogisticRegression(max_iter=1000, random_state=RNG_SEED))
])

pipe_costsens = SkPipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RNG_SEED,
                                class_weight="balanced"))
])

# Entrenamos los cuatro sobre el mismo split
estrategias = {
    "Baseline":       pipe_baseline,
    "SMOTE":          pipe_smote,
    "ADASYN":         pipe_adasyn,
    "Cost-sensitive": pipe_costsens,
}

resultados = []
for nombre, pipe in estrategias.items():
    pipe.fit(X_train, y_train)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    resultados.append({
        "Estrategia": nombre,
        "AUC-ROC": roc_auc_score(y_test, y_proba),
        "AP": average_precision_score(y_test, y_proba),
        "Recall (clase 1)": recall_score(y_test, y_pred, zero_division=0),
        "Precision (clase 1)": precision_score(y_test, y_pred, zero_division=0),
        "F2": fbeta_score(y_test, y_pred, beta=2, zero_division=0),
    })

df_res = pd.DataFrame(resultados)
print("Comparacion de estrategias para desbalanceo:\n")
print(df_res.round(4).to_string(index=False))

In [ ]:
# --- Visualización: curvas PR comparadas -----------------------------
fig, ax = plt.subplots(figsize=(11, 6))
colores = {"Baseline": COL_NAVY, "SMOTE": COL_TEAL,
           "ADASYN": COL_PURPLE, "Cost-sensitive": COL_AMBER}

for nombre, pipe in estrategias.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax.plot(rec, prec, lw=2.2, color=colores[nombre],
            label=f"{nombre} (AP = {ap:.3f})")

ax.axhline(y_test.mean(), color=COL_RED, linestyle="--", lw=1,
           label=f"Linea base = {y_test.mean():.3f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Curvas Precision-Recall: comparacion de estrategias",
             color=COL_NAVY, fontweight="bold")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

### 📝 Interpretación

La inspección conjunta de la tabla y el gráfico revela patrones típicos:

- **SMOTE y ADASYN** tienden a mejorar el recall —detectan más positivos— a costa de cierta precisión. La red neuronal conceptual: inyectamos positivos sintéticos, el modelo aprende a ser más sensible.
- **Cost-sensitive** (`class_weight='balanced'`) logra efectos similares sin alterar la geometría del dataset, simplemente reponderando el término de pérdida. Es la estrategia menos invasiva y la más recomendable como primera línea.
- El **F2-score** (que penaliza falsos negativos con peso doble) suele favorecer las estrategias de remuestreo porque su función objetivo coincide con el foco en recall.

> ⚠️ **Advertencia crítica del apunte**: SMOTE debe aplicarse **dentro del pipeline**, nunca sobre todo el dataset antes del split. Lo demostraremos operativamente en la Parte III con un ejemplo de leakage.

---

# 🔓 PARTE III — DATA LEAKAGE

> *"Un modelo con AUC de 0.99 en validación que colapsa a 0.55 en el mundo real no padece sobreajuste en el sentido clásico; padece leakage."* (Apunte, §9)

---

## 9. Cuatro ejemplos controlados de data leakage

Construimos cuatro instancias pedagógicas del fenómeno, cada una con su versión correcta comparativa. El objetivo no es solo detectar leakage: es **cuantificar la inflación artificial** que produce sobre las métricas.

### 9.1 Target leakage: variable futura

Un analista construye un modelo de default incluyendo como predictor `dias_atraso_max`, variable que sólo se conoce **después** de que el crédito evolucione. Al momento de la decisión crediticia, el banco no dispone de ella. El modelo alcanza AUC=0.98 en validación; en producción, la variable no existe.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9.1 TARGET LEAKAGE: VARIABLE FUTURA
# ══════════════════════════════════════════════════════════════

# Construimos una variable "dias_atraso_max" altamente correlacionada con
# el default (casi una función determinista: default=1 -> muchos días,
# default=0 -> pocos días). Esta variable es lo que se llama "leakage puro":
# solo se materializa después de observar y.
rng_leak = np.random.default_rng(RNG_SEED + 30)
dias_atraso = np.where(
    y == 1,
    rng_leak.integers(60, 180, size=len(y)),   # defaults: muchos dias de atraso
    rng_leak.integers(0, 15, size=len(y))      # buenos pagadores: pocos
)

X_leak = X_enc.copy()
X_leak["dias_atraso_max"] = dias_atraso

# Split limpio
X_tr, X_te, y_tr, y_te = train_test_split(
    X_leak, y, test_size=0.3, random_state=RNG_SEED, stratify=y)

# --- Modelo CON leakage (incluye dias_atraso_max) --------------------
mod_con = LogisticRegression(max_iter=1000, random_state=RNG_SEED)
mod_con.fit(X_tr, y_tr)
auc_con = roc_auc_score(y_te, mod_con.predict_proba(X_te)[:, 1])

# --- Modelo SIN leakage (excluye dias_atraso_max) --------------------
X_tr_limpio = X_tr.drop(columns=["dias_atraso_max"])
X_te_limpio = X_te.drop(columns=["dias_atraso_max"])
mod_sin = LogisticRegression(max_iter=1000, random_state=RNG_SEED)
mod_sin.fit(X_tr_limpio, y_tr)
auc_sin = roc_auc_score(y_te, mod_sin.predict_proba(X_te_limpio)[:, 1])

print("Target leakage: comparacion de AUC\n")
print(f"   CON dias_atraso_max: AUC = {auc_con:.4f}")
print(f"   SIN dias_atraso_max: AUC = {auc_sin:.4f}")
print(f"   Inflacion artificial: {auc_con - auc_sin:+.4f}")

In [ ]:
# --- Importancia de features con y sin leakage -----------------------
# Entrenamos un Random Forest que expone importancias interpretables
rf = RandomForestClassifier(n_estimators=200, random_state=RNG_SEED, n_jobs=-1)
rf.fit(X_tr, y_tr)

importancias = pd.Series(rf.feature_importances_,
                          index=X_tr.columns).sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 4.5))
colores_imp = [COL_RED if f == "dias_atraso_max" else COL_TEAL
               for f in importancias.index]
ax.barh(importancias.index, importancias.values, color=colores_imp, alpha=0.85)
ax.set_xlabel("Importancia")
ax.set_title("Top 10 features por importancia  (variable leakage en rojo)",
             color=COL_NAVY, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

print(f"\nImportancia de dias_atraso_max: {importancias.get('dias_atraso_max', 0):.4f}")
print(f"Maxima importancia del resto  : "
      f"{importancias.drop('dias_atraso_max').iloc[0]:.4f}")
print(f"Ratio: {importancias.get('dias_atraso_max', 0) / importancias.drop('dias_atraso_max').iloc[0]:.1f}x")

### 📝 Interpretación

Dos señales de alerta del apunte se activan simultáneamente: (1) el AUC supera holgadamente 0.95, y (2) una sola variable concentra una fracción desproporcionada de la importancia. Cuando estas dos señales coexisten, el diagnóstico operativo es **casi siempre target leakage**.

La cura es doble: **eliminar la variable ofensora** del dataset y **auditar todo el resto** preguntándose por cada una: *¿estaría esta información disponible al momento de tomar la decisión en producción?*

### 9.2 Train-test contamination: estandarización global

Un error clásico: estandarizar el dataset completo **antes** del `train_test_split`. La media y la desviación usadas ya incorporan información del test. El efecto operativo es modesto en este dataset pero **operacional y regulatoriamente es un leakage**: viola la ley de oro *"separar antes de transformar"*.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9.2 TRAIN-TEST CONTAMINATION: ESTANDARIZACIÓN GLOBAL
# ══════════════════════════════════════════════════════════════

from sklearn.base import clone

# --- Versión INCORRECTA: estandarizar antes de separar ---------------
scaler_global = StandardScaler()
X_glob_scaled = scaler_global.fit_transform(X_enc)  # <-- usa TODO el dataset
X_tr_mal, X_te_mal, y_tr_mal, y_te_mal = train_test_split(
    X_glob_scaled, y, test_size=0.3, random_state=RNG_SEED, stratify=y)
mod_mal = LogisticRegression(max_iter=1000, random_state=RNG_SEED)
mod_mal.fit(X_tr_mal, y_tr_mal)
auc_mal = roc_auc_score(y_te_mal, mod_mal.predict_proba(X_te_mal)[:, 1])

# --- Versión CORRECTA: pipeline que encapsula el scaler --------------
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_enc, y, test_size=0.3, random_state=RNG_SEED, stratify=y)
pipe_ok = SkPipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RNG_SEED))
])
pipe_ok.fit(X_tr2, y_tr2)  # <-- scaler.fit solo sobre X_tr2
auc_ok = roc_auc_score(y_te2, pipe_ok.predict_proba(X_te2)[:, 1])

print("Train-test contamination via estandarizacion global\n")
print(f"   INCORRECTO (fit scaler sobre todo el dataset): AUC = {auc_mal:.4f}")
print(f"   CORRECTO   (pipeline con fit solo sobre train): AUC = {auc_ok:.4f}")
print(f"\n   Diferencia: {auc_mal - auc_ok:+.4f}")

### 📝 Interpretación

La diferencia numérica aquí es pequeña (en este dataset las escalas no varían dramáticamente entre train y test), lo que hace que este tipo de leakage sea **peligrosamente invisible**. Pero la violación de principio existe: el test dejó de ser independiente. En datasets más grandes o con mayor heterogeneidad (series financieras, datos de alta dimensión) el impacto numérico es mucho mayor.

El remedio sistemático es **encapsular toda transformación en un `Pipeline`**. Así `scaler.fit()` solo ve el train y el test queda intocado hasta el momento de la predicción.

### 9.3 SMOTE aplicado antes del split

Error frecuente en tutoriales informales: aplicar SMOTE sobre todo el dataset **antes** de separar train y test. Las instancias sintéticas se generan usando vecinos que pueden pertenecer a ambos conjuntos, filtrando información del test al train.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9.3 SMOTE ANTES DEL SPLIT (error frecuente)
# ══════════════════════════════════════════════════════════════

# Usamos el escenario severo para que el efecto se aprecie
X_sev_arr = X_sev.values

# --- Versión INCORRECTA: SMOTE primero, luego split ------------------
smote_global = SMOTE(random_state=RNG_SEED, k_neighbors=3)
X_sm_global, y_sm_global = smote_global.fit_resample(X_sev_arr, y_sev)
X_tr_mal, X_te_mal, y_tr_mal, y_te_mal = train_test_split(
    X_sm_global, y_sm_global, test_size=0.3, random_state=RNG_SEED,
    stratify=y_sm_global)

mod_mal = LogisticRegression(max_iter=1000, random_state=RNG_SEED)
mod_mal.fit(X_tr_mal, y_tr_mal)
# Evaluamos sobre el test contaminado (que contiene sinteticos)
auc_mal_contaminado = roc_auc_score(
    y_te_mal, mod_mal.predict_proba(X_te_mal)[:, 1])

# --- Versión CORRECTA: split primero, SMOTE dentro del pipeline ------
X_tr_ok, X_te_ok, y_tr_ok, y_te_ok = train_test_split(
    X_sev_arr, y_sev, test_size=0.3, random_state=RNG_SEED, stratify=y_sev)
pipe_smote_ok = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RNG_SEED, k_neighbors=3)),
    ("clf", LogisticRegression(max_iter=1000, random_state=RNG_SEED))
])
pipe_smote_ok.fit(X_tr_ok, y_tr_ok)
# Evaluamos sobre el test ORIGINAL (sin contaminar)
auc_ok = roc_auc_score(y_te_ok, pipe_smote_ok.predict_proba(X_te_ok)[:, 1])

print("SMOTE antes del split: leakage de datos sinteticos\n")
print(f"   INCORRECTO (SMOTE antes de split, test contaminado): "
      f"AUC = {auc_mal_contaminado:.4f}")
print(f"   CORRECTO   (SMOTE dentro del pipeline, test limpio): "
      f"AUC = {auc_ok:.4f}")
print(f"\n   Inflacion por contaminacion: {auc_mal_contaminado - auc_ok:+.4f}")

### 📝 Interpretación

El AUC evaluado sobre el test contaminado con sintéticos es sistemáticamente más alto que el evaluado sobre test limpio. La razón técnica: los vecinos usados para interpolar sintéticos pueden venir del test, así que cuando el modelo predice sobre esos sintéticos está prediciendo sobre casos de los que ya "vio una versión cercana" durante el entrenamiento.

El patrón correcto es universal: **todo resampler debe vivir dentro del pipeline de imblearn, nunca antes del split**.

### 9.4 Temporal leakage: KFold barajado en datos con drift

Cuando los datos tienen estructura temporal y drift real, usar `KFold(shuffle=True)` en lugar de `TimeSeriesSplit` mezcla observaciones de épocas distintas en train y test, inflando las métricas.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9.4 TEMPORAL LEAKAGE: KFOLD CON DRIFT
# ══════════════════════════════════════════════════════════════

# Construimos un dataset con drift REAL entre periodos:
# la relación entre credit_amount y default se invierte en el último trimestre.
rng_t = np.random.default_rng(RNG_SEED + 40)
n_t = 1500

# Generamos datos en cuatro "trimestres" con una relación que cambia
t_labels = np.repeat(np.arange(4), n_t // 4)
x1 = rng_t.uniform(0, 10, size=n_t)
x2 = rng_t.normal(0, 1, size=n_t)

# En los trimestres 0-2: y depende positivamente de x1
# En el trimestre 3: la relación se invierte
y_drift = np.empty(n_t, dtype=int)
for i in range(n_t):
    if t_labels[i] < 3:
        logit = -2 + 0.5 * x1[i] + 0.3 * x2[i]
    else:
        logit = -2 - 0.5 * x1[i] + 0.3 * x2[i]   # invertido
    p = 1 / (1 + np.exp(-logit))
    y_drift[i] = rng_t.binomial(1, p)

X_drift = pd.DataFrame({"x1": x1, "x2": x2, "tiempo": t_labels})

# --- KFold barajado (INCORRECTO con drift) ---------------------------
mod = LogisticRegression(max_iter=1000, random_state=RNG_SEED)
auc_kfold = cross_val_score(
    mod, X_drift[["x1", "x2"]], y_drift,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED),
    scoring="roc_auc"
).mean()

# --- TimeSeriesSplit (CORRECTO) --------------------------------------
X_drift_sorted = X_drift.sort_values("tiempo").reset_index(drop=True)
y_drift_sorted = y_drift[np.argsort(t_labels)]

auc_tscv = cross_val_score(
    mod, X_drift_sorted[["x1", "x2"]], y_drift_sorted,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="roc_auc"
).mean()

print("Temporal leakage con drift real\n")
print(f"   INCORRECTO (KFold shuffle=True) : AUC = {auc_kfold:.4f}")
print(f"   CORRECTO   (TimeSeriesSplit)    : AUC = {auc_tscv:.4f}")
print(f"\n   Inflacion por ignorar temporalidad: {auc_kfold - auc_tscv:+.4f}")

### 📝 Interpretación

En este dataset sintético con drift deliberado, el KFold barajado reporta un AUC sustantivamente superior al de TimeSeriesSplit. La razón: al barajar, cada fold de validación contiene observaciones de los trimestres 0–2 (fácilmente predecibles por el modelo entrenado sobre el resto) intercaladas con observaciones del trimestre 3 (el de la regla invertida). El TimeSeriesSplit, en cambio, expone al modelo solo a observaciones pasadas durante el entrenamiento, así que cuando llega al trimestre 3 se enfrenta al drift sin haberlo visto, y su rendimiento real cae.

El TimeSeriesSplit ofrece una estimación **honesta** del rendimiento que el modelo tendría en producción futura.

## 10. El pipeline canónico blindado contra leakage

Sintetizamos todas las lecciones en un pipeline final que combina: (1) separación temporal de datos, (2) pipeline encapsulado con scaler, (3) SMOTE dentro del pipeline, (4) validación con TimeSeriesSplit, (5) evaluación sobre holdout temporal posterior. **Este es el patrón de referencia del apunte (Notas metodológicas R1–R6).**

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. PIPELINE CANÓNICO BLINDADO CONTRA LEAKAGE
# ══════════════════════════════════════════════════════════════

# (R1) SEPARAR ANTES DE TRANSFORMAR: split temporal primero
#       Los dos primeros trimestres al train, los dos ultimos al holdout
mask_dev  = periodo_col.isin(["2024Q1", "2024Q2"]).values
mask_prod = periodo_col.isin(["2024Q3", "2024Q4"]).values

X_dev,  y_dev  = X_enc[mask_dev].values,  y[mask_dev]
X_prod, y_prod = X_enc[mask_prod].values, y[mask_prod]

print(f"Desarrollo: {len(X_dev)} registros")
print(f"Produccion: {len(X_prod)} registros")
print(f"Tasa default dev:  {y_dev.mean():.1%}")
print(f"Tasa default prod: {y_prod.mean():.1%}")

# (R2) PIPELINE ENCAPSULADO
#       Scaler + SMOTE + Clasificador en una sola unidad de fit/predict
pipeline_canonico = ImbPipeline([
    ("scaler",    StandardScaler()),
    ("smote",     SMOTE(random_state=RNG_SEED, k_neighbors=5)),
    ("clf",       LogisticRegression(max_iter=1000,
                                      random_state=RNG_SEED,
                                      class_weight=None))   # SMOTE equilibra
])

# (R5) VALIDACION TEMPORAL interna usando TimeSeriesSplit
#       sobre el conjunto de desarrollo.
tscv = TimeSeriesSplit(n_splits=4)
cv_aucs = cross_val_score(pipeline_canonico, X_dev, y_dev,
                           cv=tscv, scoring="roc_auc")
print(f"\nCV temporal (4 folds): AUC = {cv_aucs.mean():.4f} +/- {cv_aucs.std():.4f}")
print(f"Folds individuales: {np.round(cv_aucs, 4)}")

# Ajuste final sobre todo el desarrollo
pipeline_canonico.fit(X_dev, y_dev)

# (R3) AUDITORIA: evaluacion honesta sobre el holdout temporal
y_proba_prod = pipeline_canonico.predict_proba(X_prod)[:, 1]
y_pred_prod  = pipeline_canonico.predict(X_prod)

auc_prod = roc_auc_score(y_prod, y_proba_prod)
ap_prod  = average_precision_score(y_prod, y_proba_prod)

print(f"\nHoldout temporal (produccion simulada):")
print(f"   AUC-ROC           = {auc_prod:.4f}")
print(f"   Average Precision = {ap_prod:.4f}")
print(f"   Gini (2*AUC-1)    = {2*auc_prod - 1:.4f}")

In [ ]:
# --- Reporte de clasificación + visualización final ------------------
print("Reporte de clasificacion sobre holdout:\n")
print(classification_report(y_prod, y_pred_prod, digits=3,
                             target_names=["No default", "Default"]))

# Curvas ROC y PR finales
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fpr, tpr, _ = roc_curve(y_prod, y_proba_prod)
axes[0].plot(fpr, tpr, color=COL_TEAL, lw=2.5,
             label=f"Pipeline canonico (AUC = {auc_prod:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color=COL_NAVY, lw=1, label="Azar")
axes[0].fill_between(fpr, tpr, alpha=0.15, color=COL_TEAL)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC sobre holdout temporal",
                  color=COL_NAVY, fontweight="bold")
axes[0].legend()

prec, rec, _ = precision_recall_curve(y_prod, y_proba_prod)
axes[1].plot(rec, prec, color=COL_PURPLE, lw=2.5,
             label=f"Pipeline canonico (AP = {ap_prod:.3f})")
axes[1].axhline(y_prod.mean(), color=COL_RED, linestyle="--", lw=1,
                label=f"Tasa base = {y_prod.mean():.3f}")
axes[1].fill_between(rec, prec, alpha=0.15, color=COL_PURPLE)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("PR sobre holdout temporal",
                  color=COL_NAVY, fontweight="bold")
axes[1].legend()

plt.tight_layout(); plt.show()

### 📝 Interpretación

El pipeline final respeta las seis reglas de oro del apunte:

- **R1 (separar antes de transformar)**: el split temporal es la primera operación.
- **R2 (pipeline encapsulado)**: scaler + SMOTE + clasificador viven en un objeto único; `fit()` solo toca el desarrollo.
- **R3 (auditoría temporal)**: el holdout (trimestres Q3–Q4) jamás entra al fit.
- **R5 (validación temporal)**: usamos `TimeSeriesSplit`, no `KFold` aleatorio.

Las métricas finales son **realistas**: un AUC típicamente entre 0.65 y 0.75 en el holdout temporal del German Credit es honesto. Si hubiéramos reportado 0.95 tendríamos un problema: un AUC tan alto en scoring crediticio sin justificación causal es una señal roja, prácticamente siempre indicativa de leakage.

---

## 11. Síntesis: tres amenazas, un objetivo común

Recuperamos la tabla comparativa final del apunte:

| Dimensión | Sesgo temporal | Desbalanceo | Data leakage |
|---|---|---|---|
| **Naturaleza** | Cambio distribucional en el tiempo | Asimetría de clases inherente | Filtración de información no disponible |
| **Síntoma** | Degradación en producción | Métricas globales engañosas | Métricas irrealistas en desarrollo |
| **Detección** | PSI, tests de drift, monitoreo | IR, curva PR, F-beta | Auditoría temporal, brecha val/prod |
| **Tratamiento** | Reentrenamiento, ventanas móviles | SMOTE, cost-sensitive, umbrales | Pipelines, partición temporal |
| **Unidad del curso** | U4 (validación), U5 (monitoreo) | U4 (remuestreo) | U2 (pipelines), U3 (modelado) |

### Tres aprendizajes centrales

1. **Las tres amenazas se interactúan.** Un modelo con leakage produce AUC inflado que enmascara el desbalanceo y el drift. Un modelo sin monitoreo sufre drift silencioso. Un modelo evaluado con accuracy oculta su incapacidad de detectar la minoritaria. **La integridad del pipeline analítico, de extremo a extremo, es la única garantía de validez.**

2. **Las métricas deben corresponder al problema.** Accuracy para problemas balanceados; AUC-ROC y Gini para scoring crediticio; Average Precision y F-beta para detección de fraude. Una métrica elegida sin considerar la estructura del problema oculta más de lo que revela.

3. **El pipeline es el lugar donde vive la disciplina metodológica.** Un `sklearn.Pipeline` bien diseñado impide técnicamente casi todas las formas de train-test contamination. Combinado con `TimeSeriesSplit` y con auditoría temporal de features, cierra la mayoría de los vectores de leakage antes de que ocurran.

### Próximos pasos en el curso

- **Clase 4**: feature engineering, selección de variables con VIF y construcción formal del scorecard.
- **Unidad 4**: tratamiento profundo de desbalanceo (SMOTE variantes, undersampling, cost-sensitive loss funcions) y validación cruzada avanzada.
- **Unidad 5**: monitoreo en producción con PSI secuencial, detección automatizada de drift con `evidently`, y protocolos de reentrenamiento.

---

## 12. Referencias bibliográficas

1. Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M. & Bouchachia, A. (2014). *A Survey on Concept Drift Adaptation*. **ACM Computing Surveys**, 46(4), 1–37.
2. Webb, G. I., Hyde, R., Cao, H., Nguyen, H. L. & Petitjean, F. (2016). *Characterizing Concept Drift*. **Data Mining and Knowledge Discovery**, 30(4), 964–994.
3. He, H. & Garcia, E. A. (2009). *Learning from Imbalanced Data*. **IEEE TKDE**, 21(9), 1263–1284.
4. Chawla, N. V., Bowyer, K. W., Hall, L. O. & Kegelmeyer, W. P. (2002). *SMOTE: Synthetic Minority Over-sampling Technique*. **JAIR**, 16, 321–357.
5. Kaufman, S., Rosset, S., Perlich, C. & Stitelman, O. (2012). *Leakage in Data Mining: Formulation, Detection, and Avoidance*. **ACM TKDD**, 6(4), 1–21.
6. Fawcett, T. (2006). *An Introduction to ROC Analysis*. **Pattern Recognition Letters**, 27(8), 861–874.
7. Siddiqi, N. (2012). *Credit Risk Scorecards*. Wiley.
8. Díaz, D. E. (2026). *Apunte Teórico Clase 2 — Sesgo Temporal, Desbalanceo de Clases y Data Leakage*. MMD, UTN FRRo.

---

<div align="center">

**Dr. Darío Ezequiel Díaz** · Aplicaciones de Minería de Datos a Economía y Finanzas
Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

</div>